# 4. Exploratory Data Analysis (EDA)

## Objetivo
Descubrir patrones, relaciones y características en los datos limpios que nos permitan:
- Entender el comportamiento del tráfico en Guadalajara
- Identificar variables predictivas para pre-colapso
- Descubrir correlaciones entre variables
- Informar la construcción de la métrica TSI

## Entrada
Datos validados de `data/processed/traffic_data_clean.csv`

## Salida
Insights y visualizaciones para feature engineering

## 4.1 Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Configurar estilo de gráficos
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print('✅ Librerías importadas y configuradas')

## 4.2 Cargar datos

In [ ]:
# Cargar datos limpios
CLEAN_PATH = '../data/processed/traffic_data_clean.csv'
df = pd.read_csv(CLEAN_PATH, parse_dates=['timestamp'])

print(f'✅ Datos cargados: {len(df)} registros')
print(f'📊 Período: {df["timestamp"].min()} a {df["timestamp"].max()}')

## 4.3 Estadísticas generales

In [ ]:
print('ESTADÍSTICAS GENERALES:\n')

print(f'Total de avenidas: {df["avenida"].nunique()}')
print(f'Total de registros: {len(df)}')
print(f'\nRegistros por avenida:')
print(df['avenida'].value_counts())

print(f'\n\nESTADÍSTICAS DE VARIABLES:')
print(df[['velocidad', 'densidad', 'detenciones']].describe().round(2))

## 4.4 Análisis por avenida

In [ ]:
print('ANÁLISIS POR AVENIDA:\n')

# Agrupar por avenida
stats_avenida = df.groupby('avenida').agg({
    'velocidad': ['mean', 'std', 'min', 'max'],
    'densidad': ['mean', 'std'],
    'detenciones': ['mean', 'std'],
    'avenida': 'count'
}).round(2)

stats_avenida.columns = ['Vel_Prom', 'Vel_Std', 'Vel_Min', 'Vel_Max', 
                         'Dens_Prom', 'Dens_Std', 'Det_Prom', 'Det_Std', 'Registros']

print(stats_avenida)

# Avenidas con menor velocidad promedio (más congestionadas)
avenidas_criticas = stats_avenida.nsmallest(3, 'Vel_Prom')
print(f'\n🔴 AVENIDAS CRÍTICAS (Menor velocidad promedio):')
for avenida in avenidas_criticas.index:
    vel = avenidas_criticas.loc[avenida, 'Vel_Prom']
    print(f'  - {avenida}: {vel} km/h promedio')

## 4.5 Visualización 1: Distribución de velocidad por avenida

In [ ]:
# Boxplot de velocidad por avenida
plt.figure(figsize=(14, 6))

sns.boxplot(data=df, x='avenida', y='velocidad', palette='Set2')
plt.title('Distribución de Velocidad por Avenida', fontsize=14, fontweight='bold')
plt.xlabel('Avenida')
plt.ylabel('Velocidad (km/h)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/eda_velocidad_por_avenida.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Gráfico guardado: eda_velocidad_por_avenida.png')

## 4.6 Análisis temporal (por horario si disponible)

In [ ]:
# Si existe columna de horario, analizar por franja
if 'horario' in df.columns:
    print('ANÁLISIS POR HORARIO:\n')
    
    stats_horario = df.groupby('horario').agg({
        'velocidad': ['mean', 'std'],
        'densidad': 'mean',
        'detenciones': 'mean',
        'horario': 'count'
    }).round(2)
    
    print(stats_horario)
    
else:
    # Crear variable de hora a partir del timestamp
    if 'timestamp' in df.columns:
        df['hora'] = df['timestamp'].dt.hour
        df['rango_horario'] = pd.cut(df['hora'], bins=[0, 8, 12, 16, 20, 24], 
                                     labels=['Madrugada', 'Mañana', 'Tarde', 'Noche', 'Noche Tard'])
        
        print('ANÁLISIS POR RANGO HORARIO:\n')
        
        stats_horario = df.groupby('rango_horario').agg({
            'velocidad': ['mean', 'std'],
            'densidad': 'mean',
            'detenciones': 'mean',
            'hora': 'count'
        }).round(2)
        
        print(stats_horario)

## 4.7 Correlación entre variables

In [ ]:
print('MATRIZ DE CORRELACIÓN:\n')

# Seleccionar columnas numéricas
columnas_numericas = ['velocidad', 'densidad', 'detenciones']
if 'latitud' in df.columns:
    columnas_numericas.extend(['latitud', 'longitud'])

correlacion = df[columnas_numericas].corr()
print(correlacion.round(3))

# Interpretar correlaciones clave
print('\n📊 CORRELACIONES CLAVE:')
print(f'  Velocidad vs Densidad: {correlacion.loc["velocidad", "densidad"]:.3f}')
print(f'  Velocidad vs Detenciones: {correlacion.loc["velocidad", "detenciones"]:.3f}')
print(f'  Densidad vs Detenciones: {correlacion.loc["densidad", "detenciones"]:.3f}')

if correlacion.loc['velocidad', 'densidad'] < -0.5:
    print('\n💡 INSIGHT: Correlación fuerte negativa entre velocidad y densidad')
    print('   → A mayor densidad, menor velocidad (como se espera en tráfico congestionado)')

## 4.8 Visualización 2: Matriz de correlación

In [ ]:
plt.figure(figsize=(10, 8))

# Heatmap de correlación
sns.heatmap(correlacion, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={'label': 'Correlación'})

plt.title('Matriz de Correlación - Variables de Tráfico', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/eda_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Gráfico guardado: eda_correlacion.png')

## 4.9 Análisis de velocidad vs densidad

In [ ]:
plt.figure(figsize=(12, 6))

# Scatter plot: Velocidad vs Densidad
scatter = plt.scatter(df['densidad'], df['velocidad'], 
                      c=df['detenciones'], cmap='YlOrRd', 
                      s=100, alpha=0.6, edgecolors='black', linewidth=0.5)

plt.colorbar(scatter, label='Detenciones')
plt.xlabel('Densidad', fontsize=12)
plt.ylabel('Velocidad (km/h)', fontsize=12)
plt.title('Relación: Velocidad vs Densidad (coloreado por Detenciones)', 
          fontsize=14, fontweight='bold')

# Agregar línea de tendencia
z = np.polyfit(df['densidad'], df['velocidad'], 1)
p = np.poly1d(z)
densidad_range = np.linspace(df['densidad'].min(), df['densidad'].max(), 100)
plt.plot(densidad_range, p(densidad_range), 'r--', linewidth=2, label='Tendencia')

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('../data/processed/eda_velocidad_densidad.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Gráfico guardado: eda_velocidad_densidad.png')

## 4.10 Identificación de estados de tráfico

In [ ]:
print('CLASIFICACIÓN DE ESTADOS DE TRÁFICO:\n')

# Clasificar estados basado en velocidad
def clasificar_estado(velocidad):
    if velocidad >= 50:
        return 'Flujo Libre'
    elif velocidad >= 30:
        return 'Carga Moderada'
    elif velocidad >= 15:
        return 'Congestión'
    else:
        return 'Congestión Severa'

df['estado_trafico'] = df['velocidad'].apply(clasificar_estado)

# Contar por estado
distribucion_estados = df['estado_trafico'].value_counts()
print('Distribución de estados:')
for estado, count in distribucion_estados.items():
    porcentaje = (count / len(df)) * 100
    print(f'  {estado}: {count} ({porcentaje:.1f}%)')

## 4.11 Visualización 3: Distribución de estados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#c0392b']
axes[0].pie(distribucion_estados, labels=distribucion_estados.index, autopct='%1.1f%%',
           colors=colors, startangle=90)
axes[0].set_title('Proporción de Estados de Tráfico', fontweight='bold')

# Bar plot
distribucion_estados.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Conteo de Estados de Tráfico', fontweight='bold')
axes[1].set_ylabel('Cantidad')
axes[1].set_xlabel('Estado')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/eda_estados_trafico.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Gráfico guardado: eda_estados_trafico.png')

## 4.12 Detección de condiciones de pre-colapso

In [ ]:
print('ANÁLISIS DE CONDICIONES DE PRE-COLAPSO:\n')

# Definir pre-colapso como: baja velocidad + alta densidad + detenciones
umbral_velocidad_baja = df['velocidad'].quantile(0.25)  # 25% más lento
umbral_densidad_alta = df['densidad'].quantile(0.75)   # 25% más denso

print(f'Umbrales detectados:')
print(f'  Velocidad baja (25%): {umbral_velocidad_baja:.2f} km/h')
print(f'  Densidad alta (75%): {umbral_densidad_alta:.2f}')

# Identificar pre-colapsos
precolapso = (df['velocidad'] <= umbral_velocidad_baja) & (df['densidad'] >= umbral_densidad_alta)
num_precolapsos = precolapso.sum()
porcentaje_precolapso = (num_precolapsos / len(df)) * 100

print(f'\n🔴 CONDICIONES DE PRE-COLAPSO DETECTADAS:')
print(f'  Registros: {num_precolapsos} ({porcentaje_precolapso:.1f}%)')

if num_precolapsos > 0:
    df_precolapso = df[precolapso]
    print(f'  Avenidas afectadas:')
    for avenida, count in df_precolapso['avenida'].value_counts().head(5).items():
        print(f'    - {avenida}: {count} registros')

## 4.13 Resumen de insights

In [ ]:
print('='*80)
print('RESUMEN DE INSIGHTS - ANÁLISIS EXPLORATORIO')
print('='*80)

print('\n1️⃣  HALLAZGOS PRINCIPALES:')
print(f'   - Velocidad promedio: {df["velocidad"].mean():.2f} km/h')
print(f'   - Velocidad mínima: {df["velocidad"].min():.2f} km/h (situaciones críticas)')
print(f'   - Densidad promedio: {df["densidad"].mean():.2f}')
print(f'   - Detenciones promedio: {df["detenciones"].mean():.2f} por registro')

print('\n2️⃣  AVENIDAS CRÍTICAS (Menor velocidad promedio):')
avenidas_crit = df.groupby('avenida')['velocidad'].mean().nsmallest(3)
for i, (avenida, vel) in enumerate(avenidas_crit.items(), 1):
    print(f'   {i}. {avenida}: {vel:.2f} km/h')

print('\n3️⃣  CORRELACIONES IMPORTANTES:')
print(f'   - Velocidad ↔ Densidad: {correlacion.loc["velocidad", "densidad"]:.3f} (correlación fuerte negativa)')
print(f'   - Velocidad ↔ Detenciones: {correlacion.loc["velocidad", "detenciones"]:.3f}')
print(f'   - Densidad ↔ Detenciones: {correlacion.loc["densidad", "detenciones"]:.3f}')

print('\n4️⃣  DISTRIBUCIÓN DE ESTADOS:')
for estado, count in distribucion_estados.items():
    porcentaje = (count / len(df)) * 100
    barra = '█' * int(porcentaje / 2)
    print(f'   {estado:20} {barra} {porcentaje:.1f}%')

print('\n5️⃣  CONDICIONES DE PRE-COLAPSO:')
print(f'   - Detectadas: {num_precolapsos} registros ({porcentaje_precolapso:.1f}%)')
print(f'   - Son registros con baja velocidad Y alta densidad simultáneamente')

print('\n6️⃣  IMPLICACIONES PARA TSI:')
print('   - La velocidad es la variable más discriminante')
print('   - Densidad y detenciones son indicadores complementarios')
print('   - La combinación de baja velocidad + alta densidad es señal de alerta')
print('   - Existen avenidas particularmente críticas que requieren atención')

print('\n' + '='*80)
print('✅ EDA COMPLETADO - Dataset listo para Feature Engineering')
print('='*80)

## 4.14 Exportar insights para siguiente fase

In [ ]:
# Guardar dataframe enriquecido con clasificaciones
df.to_csv('../data/processed/traffic_data_with_classifications.csv', index=False)
print('✅ Dataset enriquecido guardado: traffic_data_with_classifications.csv')

# Resumen de estadísticas por avenida para referencia
stats_avenida.to_csv('../data/processed/estadisticas_por_avenida.csv')
print('✅ Estadísticas por avenida guardadas: estadisticas_por_avenida.csv')

print('\n📊 Archivos de salida generados:')
print('  - traffic_data_with_classifications.csv')
print('  - estadisticas_por_avenida.csv')
print('  - eda_velocidad_por_avenida.png')
print('  - eda_correlacion.png')
print('  - eda_velocidad_densidad.png')
print('  - eda_estados_trafico.png')
print('\n✅ Todos listos en: data/processed/')

## 4.15 Próximos pasos

In [ ]:
print('''\n📋 FASE SIGUIENTE: FEATURE ENGINEERING & CONSTRUCCIÓN DE TSI

✅ Lo que ya tenemos:
  - Dataset limpio y validado
  - Insights sobre correlaciones
  - Identificación de condiciones críticas
  - Patrones de tráfico por avenida

🔄 Lo que viene:
  1. Feature Engineering
     - Crear variables de dinámicas (tasas de cambio)
     - Variables de riesgo (índices de probabilidad)
     - Ventanas temporales (series de tiempo)
     
  2. Construcción de métrica TSI
     - Integrar múltiples variables
     - Fórmula matemática
     - Calibración de pesos
     
  3. Validación
     - Pruebas de predictibilidad
     - Backtesting con datos históricos
     - Refinamiento

📊 La métrica TSI será capaz de:
  ✓ Identificar pre-colapsos con 3-10 minutos de anticipación
  ✓ Ser interpretable (escala 0-100)
  ✓ Adaptar se a diferentes avenidas
  ✓ Integrarse en sistemas de predicción en tiempo real
''')